In [6]:
import requests
import json
import getpass

# ============================================================
# CONNECTION
# ============================================================

BASE_URL = "https://t2d-registry.plhi.us"
PROGRAM_UID = "W3LSFZH3UDq"

USERNAME = "admin"
PASSWORD = getpass.getpass("pass:")

session = requests.Session()
session.auth = (USERNAME, PASSWORD)

# ============================================================
# GET ALL PROGRAM STAGES + DATA ELEMENTS
# ============================================================

r = session.get(
    f"{BASE_URL}/api/programStages",
    params={
        "filter": f"program.id:eq:{PROGRAM_UID}",
        "fields": (
            "id,name,"
            "programStageDataElements["
            "dataElement[id,name,valueType,aggregationType]"
            "]"
        ),
        "paging": "false"
    }
)

print("HTTP status:", r.status_code)
r.raise_for_status()

stages = r.json().get("programStages", [])

print("Total stages found:", len(stages))

# ============================================================
# KEEP ONLY THE STAGES WE NEED
# ============================================================

wanted_keywords = [
    "Wearable",
    "CGM",
    "Environment",
    "Cardiac",
    "ECG",
    "Diagnosis"
]

selected = []

for stage in stages:
    name = stage["name"]

    if any(word.lower() in name.lower() for word in wanted_keywords):
        selected.append(stage)

# ============================================================
# BUILD CLEAN UID DICTIONARY
# ============================================================

all_uids = {}

for stage in sorted(selected, key=lambda x: x["name"]):

    stage_name = stage["name"]
    stage_uid = stage["id"]

    fields = {}

    for psde in stage.get("programStageDataElements", []):
        de = psde.get("dataElement", {})

        if de.get("id"):
            fields[de.get("name", de["id"])] = {
                "uid": de["id"],
                "valueType": de.get("valueType"),
                "aggregationType": de.get("aggregationType")
            }

    all_uids[stage_name] = {
        "stage_uid": stage_uid,
        "data_elements": fields
    }

# ============================================================
# PRINT EVERYTHING
# ============================================================

print("\n" + "=" * 80)
print("STAGE + DATA ELEMENT UIDs")
print("=" * 80)

for stage_name, info in all_uids.items():

    print(f"\n### {stage_name}")
    print("Stage UID:", info["stage_uid"])

    for de_name, de_info in info["data_elements"].items():

        print(
            f"  {de_name:55s} "
            f"{de_info['uid']}   "
            f"[{de_info['valueType']}]"
        )

# ============================================================
# SAVE AS JSON
# ============================================================

output_file = "all_registry_stage_uids.json"

with open(output_file, "w") as f:
    json.dump(all_uids, f, indent=2)

print("\nSaved to:", output_file)

pass: ········


HTTP status: 200
Total stages found: 24

STAGE + DATA ELEMENT UIDs

### CGM Summary
Stage UID: Z7Gdp0CXP9K
  Sensor Wear Duration (days)                             nFX7TOrPMf7   [NUMBER]
  Glucose Reading Count                                   lMsHarAJOCB   [NUMBER]
  Average Glucose Level (mg/dL)                           U3FPuRIlPxU   [NUMBER]

### CGM – Glucose
Stage UID: SS7a20eCnBZ
  Glucose Mean                                            aEZ4bHknKN9   [NUMBER]
  Glucose Minimum                                         MFVsn5k7PeT   [NUMBER]
  Glucose Maximum                                         DPeHrdcFFeu   [NUMBER]
  Hourly Glucose Reading Count                            SE6sHxR9BQd   [INTEGER]
  Glucose Standard Deviation                              wMyU6v71gXo   [NUMBER]
  Time in Range Percent                                   chIDrmzk3Nj   [NUMBER]
  Time Above Range Percent                                pJUGzbk0n77   [NUMBER]
  Time Below Range Percent              

In [7]:
import os
import pandas as pd

ROOT = os.path.expanduser("~/AI-READI-fixed")

# Your original AI-READI clinical source directory
CLINICAL = os.path.join(ROOT, "clinical_data")

print("=" * 80)
print("ORIGINAL AI-READI CLINICAL DATA INVENTORY")
print("=" * 80)

if not os.path.exists(CLINICAL):
    print("NOT FOUND:", CLINICAL)
    raise SystemExit

files = []

for root, dirs, filenames in os.walk(CLINICAL):
    for filename in filenames:
        if filename.lower().endswith((".csv", ".tsv")):
            files.append(os.path.join(root, filename))

print(f"\nSource files found: {len(files)}\n")

for path in sorted(files):

    rel = os.path.relpath(path, ROOT)

    try:
        sep = "\t" if path.lower().endswith(".tsv") else ","
        df = pd.read_csv(path, sep=sep, low_memory=False)

        print("-" * 80)
        print(rel)
        print(f"Rows:    {len(df):,}")
        print(f"Columns: {len(df.columns):,}")
        print(f"Populated cells: {df.notna().sum().sum():,}")

        print("\nColumns:")
        for col in df.columns:
            n = df[col].notna().sum()
            print(f"  {col:45s} {n:>7,}")

    except Exception as e:
        print(f"\nERROR reading {rel}: {e}")

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

ORIGINAL AI-READI CLINICAL DATA INVENTORY

Source files found: 6

--------------------------------------------------------------------------------
clinical_data/condition_occurrence.csv
Rows:    12,375
Columns: 16
Populated cells: 198,000

Columns:
  condition_occurrence_id                        12,375
  person_id                                      12,375
  condition_concept_id                           12,375
  condition_start_date                           12,375
  condition_start_datetime                       12,375
  condition_end_date                             12,375
  condition_end_datetime                         12,375
  condition_type_concept_id                      12,375
  condition_status_concept_id                    12,375
  stop_reason                                    12,375
  provider_id                                    12,375
  visit_occurrence_id                            12,375
  visit_detail_id                                12,375
  condition_source_valu

In [3]:
!python3 count_true_schema_size.py

TRACKED ENTITY ATTRIBUTES: 10
  - AI-READI Person ID
  - Year of Birth
  - Marital Status
  - Diabetes Severity Group
  - Type 2 Diabetes Status (self-reported)
  - Pre-Diabetes Status
  - Family History - Parent T2D
  - Family History - Sibling T2D
  - Clinical Recruitment Site
  - Recommended ML Train/Val/Test Split

PROGRAM STAGES: 24

  CGM Summary                                     3 fields
  Diagnosis History                               3 fields
  Environment – Humidity                         11 fields
  Environment – NOx                               7 fields
  Wearable – SpO2                                13 fields
  ARCHIVED - Raw Sensor Reading (do not use)      3 fields
  Wearable – Stress                               6 fields
  Cardiac – 12-Lead ECG                          29 fields
  CGM – Glucose                                  16 fields
  Environment – PM1                               7 fields
  Environment – PM10                              7 fields
  Environm